# Team Season - team_player_onoff_summary

This notebook inspects the data **downloaded** from the `team_player_onoff_summary` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_player_onoff_summary`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_player_onoff_summary"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [ ]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


In [ ]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


In [ ]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


In [ ]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


In [ ]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


In [ ]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


In [ ]:
if df.empty:
    col_dist = pd.DataFrame()
else:
    col_null_pct = df.isna().mean()
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    col_bins = pd.cut(col_null_pct, bins=bins, include_lowest=True)
    col_counts = col_bins.value_counts().sort_index()
    col_dist = pd.DataFrame({
        "columns": col_counts,
        "pct_columns": (col_counts / len(col_null_pct) * 100).round(3),
    })

col_dist
